# New Section
This is a  weather prediction ml model

In [168]:
import pandas as pd
import numpy as np

load the dataset

In [169]:
df=pd.read_csv("/content/weatherHistory.csv")

check null values | remove empty rows |


In [170]:
df.isnull().sum()

,0
Formatted Date,0
Summary,0
Precip Type,517
Temperature (C),0
Apparent Temperature (C),0
Humidity,0
Wind Speed (km/h),0
Wind Bearing (degrees),0
Visibility (km),0
Loud Cover,0


##change the names of the columns so we can easyli deal in python

In [171]:
df.columns=df.columns.str.lower().str.replace(" ","_")

In [172]:
df.columns

Index(['formatted_date', 'summary', 'precip_type', 'temperature_(c)',
       'apparent_temperature_(c)', 'humidity', 'wind_speed_(km/h)',
       'wind_bearing_(degrees)', 'visibility_(km)', 'loud_cover',
       'pressure_(millibars)', 'daily_summary'],
      dtype='object')

In [173]:
df.sample()

,formatted_date,summary,precip_type,temperature_(c),apparent_temperature_(c),humidity,wind_speed_(km/h),wind_bearing_(degrees),visibility_(km),loud_cover,pressure_(millibars),daily_summary
19408,2008-12-24 16:00:00.000 +0100,Mostly Cloudy,rain,1.394444,1.394444,0.63,2.8497,220.0,10.1752,0.0,1026.52,Mostly cloudy starting in the morning.


fill the null values with sunny

In [174]:
df["precip_type"] = df["precip_type"].fillna("sunny")
df["precip_type"].value_counts()
df.dropna(inplace=True)

df["precip_type"].value_counts()

,count
precip_type,
rain,85224
snow,10712
sunny,517


In [175]:
from sklearn.utils import resample
rain_db=df[df["precip_type"]=="rain"]
snow=df[df["precip_type"]=="snow"]
sunny_db=df[df["precip_type"]=="sunny"]

sunny=resample(
    sunny_db,
    replace=True,
    n_samples=len(snow),
    random_state=42

)
rain=resample(
    rain_db,
    replace=True,
    n_samples=len(snow),
    random_state=42

)


df=pd.concat([rain,snow,sunny])
df=df.sample(frac=1,random_state=42,ignore_index=True)

In [176]:
df["precip_type"].value_counts()

,count
precip_type,
rain,10712
snow,10712
sunny,10712


In [177]:
x=df[["temperature_(c)","humidity","wind_speed_(km/h)","visibility_(km)","pressure_(millibars)"]]

In [178]:
y=df["precip_type"].map({"rain":0,"sunny":1,"snow":2})

In [179]:
from sklearn.model_selection import train_test_split,KFold,cross_val_score,GridSearchCV,RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report,accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MinMaxScaler

In [180]:
preprocessing=ColumnTransformer(
    transformers=[
        ("minmasscaler",MinMaxScaler(),x.columns),
    ],
    remainder="passthrough"
)

In [181]:
params={
    "model__max_depth":[3,5,7],
    "model__n_estimators":[100,200,300],
    "model__criterion": ["gini","entropy"]
}

In [182]:
cross=KFold(n_splits=5,random_state=42,shuffle=True)
pipeline=Pipeline(
    steps=[
        ("preprocessing",preprocessing),
        ("model",RandomForestClassifier(random_state=42))
    ]

)

In [183]:
model=RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=params,
    cv=cross,
    scoring="accuracy",
    n_iter=10,
)

In [184]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [185]:
model.fit(x_train,y_train)

RandomizedSearchCV(cv=KFold(n_splits=5, random_state=42, shuffle=True),
                   estimator=Pipeline(steps=[('preprocessing',
                                              ColumnTransformer(remainder='passthrough',
                                                                transformers=[('minmasscaler',
                                                                               MinMaxScaler(),
                                                                               Index(['temperature_(c)', 'humidity', 'wind_speed_(km/h)', 'visibility_(km)',
       'pressure_(millibars)'],
      dtype='object'))])),
                                             ('model',
                                              RandomForestClassifier(random_state=42))]),
                   param_distributions={'model__criterion': ['gini', 'entropy'],
                                        'model__max_depth': [3, 5, 7],
                                        'model__n_estimators': [100, 200, 300]},
                   scoring='accuracy')

In [186]:
model.best_params_

{'model__n_estimators': 300, 'model__max_depth': 7, 'model__criterion': 'gini'}

In [187]:
cross_val_score(pipeline,x_train,y_train)

array([0.99863866, 0.99824971, 0.99805523, 0.9986384 , 0.99805485])

In [188]:
y_pred=model.predict(x_test)

In [189]:
report=classification_report(y_test,y_pred)

In [190]:
sample_data=[[11.183333,0.96,10.4811,4.025,994.63]]
sd=pd.DataFrame(sample_data,columns=x.columns) #sunny

In [191]:
if (model.predict(sd))==0:
  print("rain")
elif (model.predict(sd))==1:
  print("sunny")
else:
  print("cloudy")

sunny


In [192]:
import joblib
file_name="Weather_prediction.joblib"
with open(file_name,"wb") as f:
  joblib.dump(model,f)

In [193]:
df[df["precip_type"]=="sunny"].sample()

,formatted_date,summary,precip_type,temperature_(c),apparent_temperature_(c),humidity,wind_speed_(km/h),wind_bearing_(degrees),visibility_(km),loud_cover,pressure_(millibars),daily_summary
3594,2016-10-29 01:00:00.000 +0200,Partly Cloudy,sunny,6.011111,3.377778,0.86,12.6707,260.0,0.0,0.0,1030.7,Mostly cloudy until evening.
